# EDA: анализ датасета ASVspoof2019 (LA)

Первичный разведочный анализ:
1. Автоматический поиск протоколов и аудиофайлов (устойчиво к разной структуре папок разных зеркал/Kaggle-версий датасета)
2. Объём данных и баланс классов (bonafide / spoof) по train/dev/eval
3. Распределение по типам атак (A01–A19)
4. Распределение длительностей аудио
5. Waveform, MFCC и мел-спектрограмма на паре примеров (bonafide vs spoof)

Если реальные данные ещё не скачаны (`data/raw` пуст) — ноутбук автоматически
переключается в **демо-режим** на небольшом синтетическом наборе, чтобы можно
было проверить, что весь пайплайн анализа рабочий, ещё до скачивания датасета.


## 1. Настройка окружения

In [ ]:
import os
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from dotenv import load_dotenv
import librosa
import librosa.display
import soundfile as sf

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

load_dotenv(dotenv_path=".env")
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

SEED = config["project"]["seed"]
np.random.seed(SEED)

RAW_DIR = Path(config["paths"]["raw_dir"])
PROCESSED_DIR = Path(config["paths"]["processed_dir"])
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = config["audio"]["sample_rate"]
N_MFCC = config["audio"]["n_mfcc"]
N_MELS = config["audio"]["n_mels"]
DURATION_SAMPLE_SIZE = config["eda"]["duration_sample_size"]
N_EXAMPLES_PER_CLASS = config["eda"]["n_examples_per_class"]
PROTOCOL_TABLE_PATH = Path(config["eda"]["protocol_table_path"])

print(f"RAW_DIR: {RAW_DIR.resolve()}")
print(f"Существует: {RAW_DIR.exists()}, файлов/папок внутри: {len(list(RAW_DIR.iterdir())) if RAW_DIR.exists() else 0}")


## 2. Поиск файлов протокола

Официальные протоколы ASVspoof2019 называются вида
`ASVspoof2019.LA.cm.<split>.<trn|trl>.txt` и содержат строки формата
`SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY` (KEY = `bonafide`/`spoof`).
Разные зеркала (в т.ч. на Kaggle) иногда меняют структуру папок, поэтому
поиск идёт рекурсивно и по нескольким возможным паттернам имени файла.


In [ ]:
SPLIT_PATTERNS = {
    "train": re.compile(r"train|trn", re.IGNORECASE),
    "dev": re.compile(r"\bdev\b", re.IGNORECASE),
    "eval": re.compile(r"\beval\b|\btrl\b(?!.*dev)", re.IGNORECASE),
}


def find_protocol_files(root: Path):
    """Ищет протокольные .txt файлы и сопоставляет их сплитам train/dev/eval."""
    found = {}
    if not root.exists():
        return found

    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            lower = name.lower()
            if lower.endswith(".txt") and ("cm" in lower or "protocol" in lower or "trn" in lower or "trl" in lower):
                candidates.append(Path(dirpath) / name)

    for path in candidates:
        name = path.name.lower()
        if "eval" in name:
            found.setdefault("eval", path)
        elif "dev" in name:
            found.setdefault("dev", path)
        elif "train" in name or "trn" in name:
            found.setdefault("train", path)

    return found


protocol_files = find_protocol_files(RAW_DIR)
if protocol_files:
    print("Найдены протоколы:")
    for split, path in protocol_files.items():
        print(f"  {split:<6} -> {path}")
else:
    print("Протоколы не найдены в RAW_DIR — либо датасет ещё не скачан, либо структура папок нестандартная.")


## 3. Разбор протоколов в единую таблицу

Автоматически определяет количество столбцов в файле (5 — с speaker_id,
4 — без него) и приводит всё к единому формату:
`split | speaker_id | filename | system_id | label`.


In [ ]:
def parse_protocol_file(path: Path, split: str) -> pd.DataFrame:
    raw = pd.read_csv(path, sep=r"\s+", header=None, engine="python")
    ncols = raw.shape[1]

    if ncols >= 5:
        df = raw.iloc[:, [0, 1, ncols - 2, ncols - 1]].copy()
        df.columns = ["speaker_id", "filename", "system_id", "label"]
    elif ncols == 4:
        df = raw.iloc[:, [0, 1, 2, 3]].copy()
        df.columns = ["filename", "_unused", "system_id", "label"]
        df["speaker_id"] = "unknown"
        df = df[["speaker_id", "filename", "system_id", "label"]]
    else:
        raise ValueError(f"Неожиданный формат протокола ({ncols} столбцов): {path}")

    df["label"] = df["label"].str.lower().str.strip()
    df["system_id"] = df["system_id"].replace("-", "bonafide_no_attack")
    df["split"] = split
    return df


protocol_dfs = []
for split, path in protocol_files.items():
    try:
        protocol_dfs.append(parse_protocol_file(path, split))
    except Exception as e:
        print(f"Не удалось разобрать протокол {path}: {e}")

if protocol_dfs:
    protocol_df = pd.concat(protocol_dfs, ignore_index=True)
    print(f"Всего строк в объединённом протоколе: {len(protocol_df)}")
    protocol_df.head()
else:
    protocol_df = pd.DataFrame(columns=["speaker_id", "filename", "system_id", "label", "split"])
    print("protocol_df пуст — данные ещё не скачаны или не найдены протоколы.")

protocol_df.head()


## 4. Демо-режим на синтетических данных (если реальных данных нет)

Если `protocol_df` пуст, генерируется небольшой синтетический набор
(синусоиды разной частоты как "bonafide", зашумлённые версии как "spoof"),
чтобы проверить весь последующий пайплайн анализа до скачивания реального датасета.
Как только реальные данные будут доступны — эта ветка просто не вызовется.


In [ ]:
DEMO_MODE = protocol_df.empty
DEMO_AUDIO_DIR = RAW_DIR / "_demo_synthetic"


def generate_synthetic_dataset(out_dir: Path, sr: int, seed: int):
    rng = np.random.default_rng(seed)
    out_dir.mkdir(parents=True, exist_ok=True)

    splits = {"train": 8, "dev": 4, "eval": 4}  # файлов на класс на сплит
    attack_ids = ["A01", "A02", "A03", "A04"]
    rows = []

    for split, n_per_class in splits.items():
        for i in range(n_per_class):
            # "bonafide": чистая синусоида + лёгкий шум
            duration = rng.uniform(2.0, 4.0)
            t = np.linspace(0, duration, int(sr * duration), endpoint=False)
            freq = rng.uniform(120, 220)
            bona = 0.3 * np.sin(2 * np.pi * freq * t) + 0.01 * rng.standard_normal(t.shape)
            bona_name = f"demo_{split}_bonafide_{i:03d}"
            sf.write(out_dir / f"{bona_name}.wav", bona.astype(np.float32), sr)
            rows.append({"speaker_id": "demo", "filename": bona_name, "system_id": "bonafide_no_attack", "label": "bonafide", "split": split})

            # "spoof": более "металлический" сигнал с гармониками + больше шума (имитация артефактов синтеза)
            attack = rng.choice(attack_ids)
            spoof = (
                0.2 * np.sin(2 * np.pi * freq * 2.0 * t)
                + 0.1 * np.sin(2 * np.pi * freq * 3.3 * t)
                + 0.05 * rng.standard_normal(t.shape)
            )
            spoof_name = f"demo_{split}_spoof_{i:03d}"
            sf.write(out_dir / f"{spoof_name}.wav", spoof.astype(np.float32), sr)
            rows.append({"speaker_id": "demo", "filename": spoof_name, "system_id": attack, "label": "spoof", "split": split})

    return pd.DataFrame(rows)


if DEMO_MODE:
    print("!!! ДЕМО-РЕЖИМ: реальные данные не найдены, работаем на синтетическом наборе.")
    print("    Скачайте датасет (шаг 7 в 01_project_setup.ipynb) и перезапустите этот ноутбук для анализа реальных данных.\n")
    protocol_df = generate_synthetic_dataset(DEMO_AUDIO_DIR, SAMPLE_RATE, SEED)
    print(f"Синтетических файлов сгенерировано: {len(protocol_df)}")
else:
    print("Работаем с реальными данными датасета.")

protocol_df.head()


## 5. Объём данных и баланс классов по train/dev/eval

In [ ]:
summary_table = (
    protocol_df
    .groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "dev", "eval"])
)
summary_table["total"] = summary_table.sum(axis=1)
if "bonafide" in summary_table.columns and "spoof" in summary_table.columns:
    summary_table["spoof_ratio_%"] = (summary_table["spoof"] / summary_table["total"] * 100).round(1)

print("Количество записей по сплитам и классам:")
summary_table


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_cols = [c for c in ("bonafide", "spoof") if c in summary_table.columns]
summary_table[plot_cols].plot(kind="bar", stacked=True, ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_title("Баланс классов bonafide / spoof по сплитам" + (" (демо-данные)" if DEMO_MODE else ""))
ax.set_xlabel("Сплит")
ax.set_ylabel("Количество файлов")
ax.legend(title="Класс")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Распределение по типам атак (A01–A19)

В train/dev обычно присутствуют только «известные» алгоритмы атак, а в eval —
преимущественно другие («неизвестные» на момент обучения) — так конкурс
проверяет обобщающую способность моделей на новые методы синтеза.


In [ ]:
spoof_only = protocol_df[protocol_df["label"] == "spoof"]
attack_table = (
    spoof_only
    .groupby(["split", "system_id"])
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "dev", "eval"])
)
print("Количество spoof-файлов по типам атак и сплитам:")
attack_table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
attack_table.T.plot(kind="bar", ax=ax)
ax.set_title("Распределение атак (A01–A19) по сплитам" + (" (демо-данные)" if DEMO_MODE else ""))
ax.set_xlabel("Тип атаки (system_id)")
ax.set_ylabel("Количество файлов")
ax.legend(title="Сплит")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Сопоставление имён файлов из протокола с реальными путями на диске

Строит индекс `имя_файла -> полный путь`, обходя `raw_dir` один раз — так
анализ работает независимо от того, как именно организованы подпапки в
конкретном зеркале датасета.


In [ ]:
AUDIO_EXTENSIONS = {".flac", ".wav", ".ogg", ".mp3"}


def build_audio_index(root: Path):
    index = {}
    if not root.exists():
        return index
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            stem, ext = os.path.splitext(name)
            if ext.lower() in AUDIO_EXTENSIONS:
                index[stem] = Path(dirpath) / name
    return index


audio_index = build_audio_index(RAW_DIR)
print(f"Проиндексировано аудиофайлов на диске: {len(audio_index)}")

protocol_df["full_path"] = protocol_df["filename"].map(audio_index)
missing = protocol_df["full_path"].isna().sum()
print(f"Не найдено на диске (нет соответствия по имени): {missing} из {len(protocol_df)}")


## 8. Распределение длительностей аудио

Длительность читается только из заголовка файла (`soundfile.info`), без
полной декодизации — быстро даже на нескольких тысячах файлов.


In [ ]:
available = protocol_df.dropna(subset=["full_path"])
sample_n = min(DURATION_SAMPLE_SIZE, len(available))
duration_sample = available.sample(n=sample_n, random_state=SEED) if sample_n > 0 else available

durations = []
for path in duration_sample["full_path"]:
    try:
        info = sf.info(str(path))
        durations.append(info.frames / info.samplerate)
    except Exception:
        continue

durations = np.array(durations)
if len(durations):
    print(f"Оценка по {len(durations)} файлам:")
    print(f"  среднее:   {durations.mean():.2f} c")
    print(f"  медиана:   {np.median(durations):.2f} c")
    print(f"  мин / макс: {durations.min():.2f} / {durations.max():.2f} c")

    plt.figure()
    plt.hist(durations, bins=30, color="#4C72B0", edgecolor="white")
    plt.title("Распределение длительности аудио" + (" (демо-данные)" if DEMO_MODE else f" (выборка {len(durations)} файлов)"))
    plt.xlabel("Длительность, с")
    plt.ylabel("Количество файлов")
    plt.tight_layout()
    plt.show()
else:
    print("Не удалось получить длительности — проверьте, что файлы найдены (см. шаг 7).")


## 9. Примеры: waveform, MFCC и мел-спектрограмма (bonafide vs spoof)


In [ ]:
def pick_examples(df: pd.DataFrame, label: str, n: int, split_priority=("dev", "train", "eval")):
    subset = df[(df["label"] == label) & df["full_path"].notna()]
    for split in split_priority:
        split_subset = subset[subset["split"] == split]
        if len(split_subset) >= n:
            return split_subset.sample(n=n, random_state=SEED)
    return subset.sample(n=min(n, len(subset)), random_state=SEED) if len(subset) else subset


bonafide_examples = pick_examples(protocol_df, "bonafide", N_EXAMPLES_PER_CLASS)
spoof_examples = pick_examples(protocol_df, "spoof", N_EXAMPLES_PER_CLASS)

print(f"Выбрано примеров: bonafide={len(bonafide_examples)}, spoof={len(spoof_examples)}")


In [ ]:
def load_audio(path: Path, sr: int):
    y, sr = librosa.load(path, sr=sr)
    return y, sr


def analyze_example(path: Path, sr: int, n_mfcc: int, n_mels: int, title: str):
    y, sr = load_audio(path, sr)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Waveform\n{title}")
    axes[0].set_xlabel("Время, с")
    axes[0].set_ylabel("Амплитуда")

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    img1 = librosa.display.specshow(mfcc, sr=sr, x_axis="time", ax=axes[1])
    axes[1].set_title(f"MFCC ({n_mfcc} коэфф.)")
    fig.colorbar(img1, ax=axes[1])

    # Mel-spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img2 = librosa.display.specshow(mel_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[2])
    axes[2].set_title("Мел-спектрограмма (дБ)")
    fig.colorbar(img2, ax=axes[2], format="%+2.0f dB")

    plt.tight_layout()
    plt.show()

    return {
        "duration_sec": len(y) / sr,
        "mfcc_mean": float(mfcc.mean()),
        "mfcc_std": float(mfcc.std()),
        "zero_crossing_rate": float(librosa.feature.zero_crossing_rate(y).mean()),
        "spectral_centroid_hz": float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
    }


example_stats = []

for _, row in bonafide_examples.iterrows():
    stats = analyze_example(row["full_path"], SAMPLE_RATE, N_MFCC, N_MELS, f"bonafide | {row['filename']}")
    stats.update({"label": "bonafide", "filename": row["filename"], "split": row["split"]})
    example_stats.append(stats)

for _, row in spoof_examples.iterrows():
    stats = analyze_example(row["full_path"], SAMPLE_RATE, N_MFCC, N_MELS, f"spoof ({row['system_id']}) | {row['filename']}")
    stats.update({"label": "spoof", "filename": row["filename"], "split": row["split"]})
    example_stats.append(stats)

pd.DataFrame(example_stats)


## 10. Сохранение сводной таблицы протокола

In [ ]:
protocol_df.drop(columns=["full_path"]).to_csv(PROTOCOL_TABLE_PATH, index=False)
print(f"Сводная таблица протокола сохранена: {PROTOCOL_TABLE_PATH.resolve()}")
print(f"Строк: {len(protocol_df)}")


## Дальнейшие шаги

1. Скачать реальный датасет (если этот ноутбук выполнялся в демо-режиме) и перезапустить для честного анализа.
2. На основе увиденного баланса классов — решить, нужен ли class weighting / балансировка при обучении.
3. Использовать `protocol_summary.csv` как единый источник разметки для последующих ноутбуков извлечения признаков и обучения моделей.
4. Реализовать бейзлайн (LFCC/CQCC + GMM либо MFCC + классический ML) — по итогам предыдущего обсуждения методов.
